# 05 — Collections Macro & Bivariate Analysis

## Objective

### Temporal rule

Portanto, o valor recuperado historicamente não deve ser dividido pelo saldo em aberto de setembro e interpretado ou rotulado como taxa de recuperação,   
pois numerador e denominador pertencem a períodos temporais distintos.

In [83]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


## 5.1 Grain validation

In [88]:
assert queue["customer_id"].notna().all()
assert queue["customer_id"].is_unique, "Queue is not 1 row per customer."
assert wa["message_id"].notna().all()
assert wa["message_id"].is_unique, "WhatsApp history is not 1 row per message."

queue_ids = set(queue["customer_id"])
wa_ids = set(wa["customer_id"])

display(pd.DataFrame({
    "metric": ["queue_rows","queue_unique_customers","wa_rows","wa_unique_customers",
               "customers_in_both","queue_customers_without_wa_history"],
    "value": [len(queue),queue["customer_id"].nunique(),len(wa),wa["customer_id"].nunique(),
              len(queue_ids & wa_ids),len(queue_ids - wa_ids)]
}))

,metric,value
0,queue_rows,10658
1,queue_unique_customers,10658
2,wa_rows,75406
3,wa_unique_customers,11724
4,customers_in_both,5382
5,queue_customers_without_wa_history,5276


,customer_id,in_collections_since,days_past_due_on_2026-09-01,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf,dpd_bucket,salary_bucket,balance_bucket,debt_to_salary,sep_entry_group,n_messages,n_failed,n_delivered,n_read,n_clicked,n_replied,n_payment_events,total_amount_paid_brl,max_observed_dpd,min_observed_dpd,first_contact_at,initial_observed_dpd,initial_observed_balance,last_contact_at,last_observed_dpd,last_observed_balance,ever_failed,ever_delivered,ever_read,ever_clicked,ever_replied,ever_paid,dpd_change_observed,observed_balance_reduction,observed_balance_reduction_pct,dpd_migration,payment_status,_merge,has_wa_history,contact_history_group,contact_history_group_analysis,message_pressure_bucket,n_messages_delivery,n_delivered_corrected,n_failed_corrected,n_failed_blocked,n_failed_invalid_number,n_failed_unreachable,entered_sep_2026_flag
0,C000002,2026-07-26,38,"1,143.79","2,670.00",20,4,7,49,BA,31–60,1.5–3k,1–2.5k,0.43,Pre-existing before Sep-2026,4.00,0.00,4.00,2.00,0.00,0.00,0.00,0.00,16.00,4.00,2026-07-29 10:31:00,4.00,"1,143.79",2026-08-10 14:53:00,16.00,"1,143.79",False,True,True,False,False,False,12.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,4.00,4,0,0,0,0,False
1,C000005,2026-08-05,28,250.35,"3,200.00",5,1,5,46,RS,16–30,3–5k,250–500,0.08,Pre-existing before Sep-2026,5.00,0.00,5.00,1.00,0.00,0.00,0.00,0.00,21.00,1.00,2026-08-05 14:36:00,1.00,250.35,2026-08-25 19:13:00,21.00,250.35,False,True,True,False,False,False,20.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,5.00,5,0,0,0,0,False
2,C000011,2026-07-13,51,509.53,"3,320.00",1,19,17,15,RS,31–60,3–5k,500–1k,0.15,Pre-existing before Sep-2026,11.00,0.00,11.00,2.00,2.00,0.00,1.00,456.62,48.00,1.00,2026-07-13 11:20:00,1.00,966.15,2026-08-29 10:23:00,48.00,509.53,False,True,True,True,False,True,47.00,456.62,0.47,Deteriorated,Partial payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,11–20,11.00,11,0,0,0,0,False
3,C000014,2026-08-02,31,"1,032.03","2,310.00",20,10,11,10,SP,31–60,1.5–3k,1–2.5k,0.45,Pre-existing before Sep-2026,5.00,0.00,5.00,0.00,2.00,1.00,0.00,0.00,23.00,2.00,2026-08-03 20:18:00,2.00,"1,032.03",2026-08-24 15:56:00,23.00,"1,032.03",False,True,False,True,True,False,21.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,5.00,5,0,0,0,0,False
4,C000015,2026-07-22,42,"1,458.43","2,580.00",30,8,3,33,GO,31–60,1.5–3k,1–2.5k,0.57,Pre-existing before Sep-2026,6.00,0.00,6.00,1.00,2.00,0.00,0.00,0.00,14.00,1.00,2026-07-22 09:46:00,1.00,"1,458.43",2026-08-04 10:58:00,14.00,"1,458.43",False,True,True,True,False,False,13.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,6–10,6.00,6,0,0,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10653,C016996,2026-09-23,0,750.27,"2,620.00",20,6,7,8,PR,Current / 0,1.5–3k,500–1k,0.29,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False,No Jun–Aug WhatsApp history,Not historically eligible,0,0.00,0,0,0,0,0,True
10654,C016997,2026-09-24,0,433.73,"2,720.00",5,28,25,27,BA,Current / 0,1.5–3k,250–500,0.16,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False,No Jun–Aug WhatsApp history,Not historically eligible,0,0.00,0,0,0,0,0,True
10655,C016998,2026-09-28,0,667.08,"6,480.00",1,5,8,24,CE,Current / 0,5–10k,500–1k,0.10,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN

## 5.1.1 População Jun-Ago / Sep

Regra de população: in_collections_since define a elegibilidade para a análise histórica. Clientes que entraram em Collections a partir de 01/09/2026 permanecem na fotografia da carteira de setembro, mas são excluídos das comparações de performance histórica de junho a agosto. A ausência de atividade histórica de WhatsApp para esses clientes é estrutural e não deve ser interpretada como “não contatado” ou “sem resposta”. Esses clientes formam uma população sem histórico observado, para a qual a estratégia aprendida com os dados históricos poderá posteriormente ser generalizada/inferida.

In [87]:
# ============================================================
# September entry population
# ============================================================

customer["in_collections_since"] = pd.to_datetime(
    customer["in_collections_since"],
    errors="coerce"
)

customer["entered_sep_2026_flag"] = (
    customer["in_collections_since"] >= pd.Timestamp("2026-09-01")
)

customer["sep_entry_group"] = np.where(
    customer["entered_sep_2026_flag"],
    "Entered in Sep-2026",
    "Pre-existing before Sep-2026"
)

customer

,customer_id,in_collections_since,days_past_due_on_2026-09-01,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf,dpd_bucket,salary_bucket,balance_bucket,debt_to_salary,sep_entry_group,n_messages,n_failed,n_delivered,n_read,n_clicked,n_replied,n_payment_events,total_amount_paid_brl,max_observed_dpd,min_observed_dpd,first_contact_at,initial_observed_dpd,initial_observed_balance,last_contact_at,last_observed_dpd,last_observed_balance,ever_failed,ever_delivered,ever_read,ever_clicked,ever_replied,ever_paid,dpd_change_observed,observed_balance_reduction,observed_balance_reduction_pct,dpd_migration,payment_status,_merge,has_wa_history,contact_history_group,contact_history_group_analysis,message_pressure_bucket,n_messages_delivery,n_delivered_corrected,n_failed_corrected,n_failed_blocked,n_failed_invalid_number,n_failed_unreachable,entered_sep_2026_flag
0,C000002,2026-07-26,38,"1,143.79","2,670.00",20,4,7,49,BA,31–60,1.5–3k,1–2.5k,0.43,Pre-existing before Sep-2026,4.00,0.00,4.00,2.00,0.00,0.00,0.00,0.00,16.00,4.00,2026-07-29 10:31:00,4.00,"1,143.79",2026-08-10 14:53:00,16.00,"1,143.79",False,True,True,False,False,False,12.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,4.00,4,0,0,0,0,False
1,C000005,2026-08-05,28,250.35,"3,200.00",5,1,5,46,RS,16–30,3–5k,250–500,0.08,Pre-existing before Sep-2026,5.00,0.00,5.00,1.00,0.00,0.00,0.00,0.00,21.00,1.00,2026-08-05 14:36:00,1.00,250.35,2026-08-25 19:13:00,21.00,250.35,False,True,True,False,False,False,20.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,5.00,5,0,0,0,0,False
2,C000011,2026-07-13,51,509.53,"3,320.00",1,19,17,15,RS,31–60,3–5k,500–1k,0.15,Pre-existing before Sep-2026,11.00,0.00,11.00,2.00,2.00,0.00,1.00,456.62,48.00,1.00,2026-07-13 11:20:00,1.00,966.15,2026-08-29 10:23:00,48.00,509.53,False,True,True,True,False,True,47.00,456.62,0.47,Deteriorated,Partial payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,11–20,11.00,11,0,0,0,0,False
3,C000014,2026-08-02,31,"1,032.03","2,310.00",20,10,11,10,SP,31–60,1.5–3k,1–2.5k,0.45,Pre-existing before Sep-2026,5.00,0.00,5.00,0.00,2.00,1.00,0.00,0.00,23.00,2.00,2026-08-03 20:18:00,2.00,"1,032.03",2026-08-24 15:56:00,23.00,"1,032.03",False,True,False,True,True,False,21.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,4–5,5.00,5,0,0,0,0,False
4,C000015,2026-07-22,42,"1,458.43","2,580.00",30,8,3,33,GO,31–60,1.5–3k,1–2.5k,0.57,Pre-existing before Sep-2026,6.00,0.00,6.00,1.00,2.00,0.00,0.00,0.00,14.00,1.00,2026-07-22 09:46:00,1.00,"1,458.43",2026-08-04 10:58:00,14.00,"1,458.43",False,True,True,True,False,False,13.00,0.00,0.00,Deteriorated,No payment,both,True,With Jun–Aug WhatsApp history,With Jun–Aug WhatsApp history,6–10,6.00,6,0,0,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10653,C016996,2026-09-23,0,750.27,"2,620.00",20,6,7,8,PR,Current / 0,1.5–3k,500–1k,0.29,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False,No Jun–Aug WhatsApp history,Not historically eligible,0,0.00,0,0,0,0,0,True
10654,C016997,2026-09-24,0,433.73,"2,720.00",5,28,25,27,BA,Current / 0,1.5–3k,250–500,0.16,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False,No Jun–Aug WhatsApp history,Not historically eligible,0,0.00,0,0,0,0,0,True
10655,C016998,2026-09-28,0,667.08,"6,480.00",1,5,8,24,CE,Current / 0,5–10k,500–1k,0.10,Entered in Sep-2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaT,NaN,NaN,NaT,NaN,NaN,NaN

In [86]:
# ============================================================
# September entry population
# ============================================================

# Ensure datetime
customer["in_collections_since"] = pd.to_datetime(
    customer["in_collections_since"],
    errors="coerce"
)

# Create population group directly in customer
customer["sep_entry_group"] = np.select(
    [
        customer["in_collections_since"].isna(),
        customer["in_collections_since"] >= pd.Timestamp("2026-09-01")
    ],
    [
        "Unknown entry date",
        "Entered in September"
    ],
    default="Pre-existing in September"
)


# ============================================================
# September entry view
# ============================================================

sep_entry_view = (
    customer
    .groupby("sep_entry_group", observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        sep_outstanding_brl=("outstanding_balance_brl", "sum"),
        avg_sep_balance_brl=("outstanding_balance_brl", "mean"),
        median_sep_balance_brl=("outstanding_balance_brl", "median"),
        avg_sep_dpd=("days_past_due_on_2026-09-01", "mean")
    )
)

sep_entry_view["customer_share"] = (
    sep_entry_view["customers"]
    / sep_entry_view["customers"].sum()
)

sep_entry_view["sep_exposure_share"] = (
    sep_entry_view["sep_outstanding_brl"]
    / sep_entry_view["sep_outstanding_brl"].sum()
)

display(sep_entry_view)

,customers,sep_outstanding_brl,avg_sep_balance_brl,median_sep_balance_brl,avg_sep_dpd,customer_share,sep_exposure_share
sep_entry_group,,,,,,,
Entered in September,5000,"4,284,358.42",856.87,747.92,0.00,0.47,0.48
Pre-existing in September,5658,"4,601,649.56",813.30,708.04,29.39,0.53,0.52


In [31]:
# Full September portfolio
portfolio_sep = customer.copy()

# Population eligible for Jun-Aug historical analysis
historical_population = customer.loc[
    ~customer["entered_sep_2026_flag"]
].copy()

# New September entrants
new_sep_entrants = customer.loc[
    customer["entered_sep_2026_flag"]
].copy()

print("=" * 70)
print("POPULATION DEFINITION")
print("=" * 70)

print(
    f"September portfolio       : "
    f"{portfolio_sep['customer_id'].nunique():,} customers"
)

print(
    f"Historical population     : "
    f"{historical_population['customer_id'].nunique():,} customers"
)

print(
    f"New September entrants    : "
    f"{new_sep_entrants['customer_id'].nunique():,} customers"
)

print("=" * 70)

POPULATION DEFINITION
September portfolio       : 10,658 customers
Historical population     : 5,658 customers
New September entrants    : 5,000 customers


## 5.2 Analytical buckets

In [32]:
def dpd_bucket(s):
    return pd.cut(s, [-np.inf,0,7,15,30,60,90,np.inf],
                  labels=["Current / 0","1–7","8–15","16–30","31–60","61–90","91+"])

queue["dpd_bucket"] = dpd_bucket(queue["days_past_due_on_2026-09-01"])
queue["salary_bucket"] = pd.cut(
    queue["monthly_salary_brl"], [-np.inf,1500,3000,5000,10000,np.inf],
    labels=["≤1.5k","1.5–3k","3–5k","5–10k",">10k"]
)
queue["balance_bucket"] = pd.cut(
    queue["outstanding_balance_brl"], [-np.inf,250,500,1000,2500,5000,np.inf],
    labels=["≤250","250–500","500–1k","1–2.5k","2.5–5k",">5k"]
)
queue["debt_to_salary"] = (
    queue["outstanding_balance_brl"] /
    queue["monthly_salary_brl"].replace(0, np.nan)
)

## 4.3 Portfolio snapshot — September

In [35]:
# ============================================================
# Portfolio KPIs — Total + September Entry Group
# ============================================================

def calculate_portfolio_kpis(df):
    return pd.Series({
        "customers": df["customer_id"].nunique(),
        "outstanding_brl": df["outstanding_balance_brl"].sum(),
        "avg_balance_brl": df["outstanding_balance_brl"].mean(),
        "median_balance_brl": df["outstanding_balance_brl"].median(),
        "avg_dpd": df["days_past_due_on_2026-09-01"].mean(),
        "median_dpd": df["days_past_due_on_2026-09-01"].median(),
        "avg_debt_to_salary": (
            df["debt_to_salary"]
            .replace([np.inf, -np.inf], np.nan)
            .mean()
        )
    })


# ------------------------------------------------------------
# Total September portfolio
# ------------------------------------------------------------

portfolio_kpis_total = (
    calculate_portfolio_kpis(customer)
    .to_frame("Total portfolio")
)


# ------------------------------------------------------------
# By September entry group
# ------------------------------------------------------------

portfolio_kpis_population = (
    customer
    .groupby("sep_entry_group", observed=False)
    .apply(
        calculate_portfolio_kpis,
        include_groups=False
    )
    .T
)


# ------------------------------------------------------------
# Final comparative view
# ------------------------------------------------------------

portfolio_kpis = pd.concat(
    [
        portfolio_kpis_total,
        portfolio_kpis_population
    ],
    axis=1
)


display(portfolio_kpis)


,Total portfolio,Entered in September,Pre-existing in September
customers,"10,658.00","5,000.00","5,658.00"
outstanding_brl,"8,886,007.98","4,284,358.42","4,601,649.56"
avg_balance_brl,833.74,856.87,813.30
median_balance_brl,727.83,747.92,708.04
avg_dpd,15.60,0.00,29.39
median_dpd,4.00,0.00,29.00
avg_debt_to_salary,0.32,0.33,0.31


## 4.4 Historical recovery — June to August

Dentro de cada cliente, valores de amount_paid_brl iguais observados em mensagens cujas janelas de 72h se sobrepõem não são somados como pagamentos independentes. Um novo pagamento só pode ser acumulado quando estiver fora da janela de 72h do pagamento anteriormente contabilizado.

In [36]:
wa = wa.sort_values(["customer_id", "sent_at"]).copy()

wa["sent_at"] = pd.to_datetime(wa["sent_at"])

def reconstruct_attributed_payments(group, window_hours=72):
    group = group.sort_values("sent_at").copy()

    last_counted_time = {}
    counted = []

    for _, row in group.iterrows():
        amount = row["amount_paid_brl"]
        sent_at = row["sent_at"]

        if pd.isna(amount) or amount <= 0:
            counted.append(0.0)
            continue

        previous_time = last_counted_time.get(amount)

        if (
            previous_time is not None
            and (sent_at - previous_time).total_seconds() <= window_hours * 3600
        ):
            # Same amount inside overlapping 72h attribution window:
            # conservatively treat as the same economic payment.
            counted.append(0.0)

        else:
            counted.append(float(amount))
            last_counted_time[amount] = sent_at

    group["deduplicated_attributed_payment"] = counted

    return group


wa = (
    wa
    .groupby("customer_id", group_keys=False)
    .apply(reconstruct_attributed_payments)
)

C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3195582472.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(reconstruct_attributed_payments)


In [37]:
payment_summary = (
    wa
    .groupby("customer_id")
    .agg(
        raw_attributed_payment_72h=("amount_paid_brl", "sum"),
        deduplicated_attributed_payment_72h=(
            "deduplicated_attributed_payment",
            "sum"
        ),
        n_payment_attributions=("amount_paid_brl", lambda x: (x > 0).sum()),
        n_messages=("customer_id", "size"),
    )
    .reset_index()
)

payment_summary["possible_payment_double_count"] = (
    payment_summary["raw_attributed_payment_72h"]
    >
    payment_summary["deduplicated_attributed_payment_72h"]
)

In [38]:
# ============================================================
# 4.4 Historical recovery — June to August
# Macro recovery view after 72h attribution deduplication
# ============================================================

historical_recovery = (
    wa
    .groupby("customer_id", as_index=False)
    .agg(
        n_messages=("customer_id", "size"),

        raw_attributed_payment_72h=(
            "amount_paid_brl",
            "sum"
        ),

        historical_recovered_brl=(
            "deduplicated_attributed_payment",
            "sum"
        ),

        n_positive_payment_attributions=(
            "amount_paid_brl",
            lambda x: (x.fillna(0) > 0).sum()
        ),

        n_deduplicated_payments=(
            "deduplicated_attributed_payment",
            lambda x: (x.fillna(0) > 0).sum()
        ),
    )
)

# ------------------------------------------------------------
# Guardrail diagnostics
# ------------------------------------------------------------

historical_recovery["potential_duplicate_attribution_brl"] = (
    historical_recovery["raw_attributed_payment_72h"]
    - historical_recovery["historical_recovered_brl"]
)

historical_recovery["possible_double_count_flag"] = (
    historical_recovery["potential_duplicate_attribution_brl"] > 0
)

historical_recovery["recovered_flag"] = (
    historical_recovery["historical_recovered_brl"] > 0
)


# ------------------------------------------------------------
# Portfolio-level macro indicators
# ------------------------------------------------------------

n_customers = historical_recovery["customer_id"].nunique()

n_recovered_customers = (
    historical_recovery["recovered_flag"].sum()
)

raw_recovery = (
    historical_recovery["raw_attributed_payment_72h"].sum()
)

deduplicated_recovery = (
    historical_recovery["historical_recovered_brl"].sum()
)

potential_duplicate = (
    historical_recovery[
        "potential_duplicate_attribution_brl"
    ].sum()
)

customers_affected = (
    historical_recovery[
        "possible_double_count_flag"
    ].sum()
)


# ------------------------------------------------------------
# Print macro recovery view
# ------------------------------------------------------------

print("=" * 70)
print("HISTORICAL RECOVERY — JUN TO AUG 2026")
print("=" * 70)

print(f"Customers in interaction history       : {n_customers:,}")
print(f"Customers with attributed recovery     : {n_recovered_customers:,}")

if n_customers > 0:
    print(
        f"Customers with recovery (%)            : "
        f"{n_recovered_customers / n_customers:.2%}"
    )

print()
print("PAYMENT ATTRIBUTION")
print("-" * 70)

print(
    f"Raw attributed payment                 : "
    f"R$ {raw_recovery:,.2f}"
)

print(
    f"Deduplicated historical recovery       : "
    f"R$ {deduplicated_recovery:,.2f}"
)

print(
    f"Potential duplicated attribution       : "
    f"R$ {potential_duplicate:,.2f}"
)

if raw_recovery > 0:
    print(
        f"Potential duplication over raw (%)     : "
        f"{potential_duplicate / raw_recovery:.2%}"
    )

print(
    f"Customers affected by guardrail        : "
    f"{customers_affected:,}"
)

if customers_affected > 0:
    print(
        f"Customers affected (%)                 : "
        f"{customers_affected / n_customers:.2%}"
    )

print()
print("RECOVERY AMONG PAYERS")
print("-" * 70)

payers = historical_recovery.loc[
    historical_recovery["historical_recovered_brl"] > 0,
    "historical_recovered_brl"
]

if len(payers) > 0:

    print(
        f"Average recovered per payer            : "
        f"R$ {payers.mean():,.2f}"
    )

    print(
        f"Median recovered per payer             : "
        f"R$ {payers.median():,.2f}"
    )

    print(
        f"P25 recovered per payer                : "
        f"R$ {payers.quantile(.25):,.2f}"
    )

    print(
        f"P75 recovered per payer                : "
        f"R$ {payers.quantile(.75):,.2f}"
    )

    print(
        f"P90 recovered per payer                : "
        f"R$ {payers.quantile(.90):,.2f}"
    )

print("=" * 70)

HISTORICAL RECOVERY — JUN TO AUG 2026
Customers in interaction history       : 11,724
Customers with attributed recovery     : 4,893
Customers with recovery (%)            : 41.73%

PAYMENT ATTRIBUTION
----------------------------------------------------------------------
Raw attributed payment                 : R$ 3,459,305.30
Deduplicated historical recovery       : R$ 3,459,305.30
Potential duplicated attribution       : R$ 0.00
Potential duplication over raw (%)     : 0.00%
Customers affected by guardrail        : 0

RECOVERY AMONG PAYERS
----------------------------------------------------------------------
Average recovered per payer            : R$ 706.99
Median recovered per payer             : R$ 582.87
P25 recovered per payer                : R$ 330.25
P75 recovered per payer                : R$ 948.51
P90 recovered per payer                : R$ 1,410.45


## 4.5 Customer-level WhatsApp history

This is the **safe bridge table**: 75k+ message events become one row per customer before the September join.

In [39]:
wa = wa.sort_values(["customer_id","sent_at","message_id"]).copy()

first_obs = wa.groupby("customer_id", as_index=False).first()[
    ["customer_id","sent_at","days_past_due","outstanding_balance_brl"]
].rename(columns={
    "sent_at":"first_contact_at",
    "days_past_due":"initial_observed_dpd",
    "outstanding_balance_brl":"initial_observed_balance"
})

last_obs = wa.groupby("customer_id", as_index=False).last()[
    ["customer_id","sent_at","days_past_due","outstanding_balance_brl"]
].rename(columns={
    "sent_at":"last_contact_at",
    "days_past_due":"last_observed_dpd",
    "outstanding_balance_brl":"last_observed_balance"
})

delivery = wa["delivery_status"].astype(str).str.lower()
interaction = wa["interaction"].astype(str).str.lower()

wa_flags = wa.assign(
    is_failed=delivery.eq("failed"),
    is_delivered=delivery.ne("failed"),
    is_read=interaction.str.contains("read", na=False),
    is_clicked=interaction.str.contains("click", na=False),
    is_replied=interaction.str.contains("repl|reply", regex=True, na=False),
    is_paid=wa["paid_within_72h"].fillna(False).astype(bool)
)

wa_customer = wa_flags.groupby("customer_id", as_index=False).agg(
    n_messages=("message_id","count"),
    n_failed=("is_failed","sum"),
    n_delivered=("is_delivered","sum"),
    n_read=("is_read","sum"),
    n_clicked=("is_clicked","sum"),
    n_replied=("is_replied","sum"),
    n_payment_events=("is_paid","sum"),
    total_amount_paid_brl=("amount_paid_brl","sum"),
    max_observed_dpd=("days_past_due","max"),
    min_observed_dpd=("days_past_due","min")
)

wa_customer = (
    wa_customer
    .merge(first_obs, on="customer_id", how="left", validate="1:1")
    .merge(last_obs, on="customer_id", how="left", validate="1:1")
)

for col in ["failed","delivered","read","clicked","replied"]:
    wa_customer[f"ever_{col}"] = wa_customer[f"n_{col}"].gt(0)
wa_customer["ever_paid"] = wa_customer["n_payment_events"].gt(0)

wa_customer["dpd_change_observed"] = (
    wa_customer["last_observed_dpd"] - wa_customer["initial_observed_dpd"]
)
wa_customer["observed_balance_reduction"] = (
    wa_customer["initial_observed_balance"] - wa_customer["last_observed_balance"]
)
wa_customer["observed_balance_reduction_pct"] = (
    wa_customer["observed_balance_reduction"] /
    wa_customer["initial_observed_balance"].replace(0,np.nan)
)

wa_customer["dpd_migration"] = np.select(
    [wa_customer["dpd_change_observed"] < 0,
     wa_customer["dpd_change_observed"] == 0,
     wa_customer["dpd_change_observed"] > 0],
    ["Improved","Stable","Deteriorated"],
    default="Unknown"
)

wa_customer["payment_status"] = np.select(
    [wa_customer["total_amount_paid_brl"].le(0),
     (wa_customer["total_amount_paid_brl"].gt(0) &
      wa_customer["total_amount_paid_brl"].lt(wa_customer["initial_observed_balance"])),
     (wa_customer["total_amount_paid_brl"].gt(0) &
      wa_customer["total_amount_paid_brl"].ge(wa_customer["initial_observed_balance"]))],
    ["No payment","Partial payment","Full / >= initial observed balance"],
    default="Unknown"
)

assert wa_customer["customer_id"].is_unique
display(wa_customer.head())
print("Customer history:", wa_customer.shape)

,customer_id,n_messages,n_failed,n_delivered,n_read,n_clicked,n_replied,n_payment_events,total_amount_paid_brl,max_observed_dpd,min_observed_dpd,first_contact_at,initial_observed_dpd,initial_observed_balance,last_contact_at,last_observed_dpd,last_observed_balance,ever_failed,ever_delivered,ever_read,ever_clicked,ever_replied,ever_paid,dpd_change_observed,observed_balance_reduction,observed_balance_reduction_pct,dpd_migration,payment_status
0,C000001,7,0,7,0,1,0,1,934.58,26,3,2026-07-06 13:29:00,3,934.58,2026-07-29 09:14:00,26,934.58,False,True,False,True,False,True,23,0.00,0.00,Deteriorated,Full / >= initial observed balance
1,C000002,4,0,4,2,0,0,0,0.00,16,4,2026-07-29 10:31:00,4,"1,143.79",2026-08-10 14:53:00,16,"1,143.79",False,True,True,False,False,False,12,0.00,0.00,Deteriorated,No payment
2,C000003,12,0,12,2,1,1,0,0.00,60,3,2026-06-09 09:52:00,3,758.22,2026-08-05 10:54:00,60,758.22,False,True,True,True,True,False,57,0.00,0.00,Deteriorated,No payment
3,C000004,5,0,5,4,0,0,1,"1,331.09",11,1,2026-06-30 17:46:00,1,"1,331.09",2026-07-10 10:57:00,11,"1,331.09",False,True,True,False,False,True,10,0.00,0.00,Deteriorated,Full / >= initial observed balance
4,C000005,5,0,5,1,0,0,0,0.00,21,1,2026-08-05 14:36:00,1,250.35,2026-08-25 19:13:00,21,250.35,False,True,True,False,False,False,20,0.00,0.00,Deteriorated,No payment


Customer history: (11724, 28)


Alerta de interpretação: initial_observed_dpd e last_observed_dpd representam o primeiro e o último DPD observados nos eventos de WhatsApp. Este é um indicador de migração observado nos momentos de contato, e ainda não uma matriz formal de roll rate mensal.

## 4.6 Safe September analytical base

In [44]:
# ============================================================
# September population definition
# BEFORE merging WhatsApp history
# ============================================================

queue["in_collections_since"] = pd.to_datetime(
    queue["in_collections_since"],
    errors="coerce"
)

queue["sep_entry_group"] = np.select(
    [
        queue["in_collections_since"].isna(),
        queue["in_collections_since"] >= pd.Timestamp("2026-09-01")
    ],
    [
        "Unknown entry date",
        "Entered in September"
    ],
    default="Pre-existing in September"
)

In [45]:
customer = queue.merge(
    wa_customer, on="customer_id", how="left", validate="1:1", indicator=True
)

customer["has_wa_history"] = customer["_merge"].eq("both")
customer["contact_history_group"] = np.where(
    customer["has_wa_history"],
    "With Jun–Aug WhatsApp history",
    "No Jun–Aug WhatsApp history"
)

count_cols = ["n_messages","n_failed","n_delivered","n_read","n_clicked",
              "n_replied","n_payment_events","total_amount_paid_brl"]
customer[count_cols] = customer[count_cols].fillna(0)

assert len(customer) == len(queue)
assert customer["customer_id"].is_unique
assert np.isclose(customer["outstanding_balance_brl"].sum(),
                  queue["outstanding_balance_brl"].sum())

print("SAFE MERGE PASSED")
print("Rows:", len(customer))
print("September outstanding:", customer["outstanding_balance_brl"].sum())

SAFE MERGE PASSED
Rows: 10658
September outstanding: 8886007.98


## 4.7 With vs without historical contact

In [47]:
# ============================================================
# Historical contact eligibility
# ============================================================

customer["contact_history_group_analysis"] = np.where(
    customer["sep_entry_group"].eq("Entered in September"),
    "Not historically eligible",
    customer["contact_history_group"].fillna("No historical contact")
)

In [48]:
# ============================================================
# Contact History View — by September Entry Group
# ============================================================

contact_view = (
    customer
    .groupby(
        [
            "sep_entry_group",
            "contact_history_group_analysis"
        ],
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        sep_outstanding_brl=("outstanding_balance_brl", "sum"),
        avg_sep_balance_brl=("outstanding_balance_brl", "mean"),
        median_sep_balance_brl=("outstanding_balance_brl", "median"),
        avg_sep_dpd=("days_past_due_on_2026-09-01", "mean"),
        historical_recovered_brl=("total_amount_paid_brl", "sum"),
        historical_payment_events=("n_payment_events", "sum")
    )
    .reset_index()
)

contact_view["customer_share"] = (
    contact_view["customers"]
    / contact_view
        .groupby("sep_entry_group")["customers"]
        .transform("sum")
)

contact_view["sep_exposure_share"] = (
    contact_view["sep_outstanding_brl"]
    / contact_view
        .groupby("sep_entry_group")["sep_outstanding_brl"]
        .transform("sum")
)

display(contact_view)

,sep_entry_group,contact_history_group_analysis,customers,sep_outstanding_brl,avg_sep_balance_brl,median_sep_balance_brl,avg_sep_dpd,historical_recovered_brl,historical_payment_events,customer_share,sep_exposure_share
0,Entered in September,Not historically eligible,5000,"4,284,358.42",856.87,747.92,0.00,0.00,0.00,1.00,1.00
1,Pre-existing in September,No Jun–Aug WhatsApp history,276,"218,985.81",793.43,659.11,4.18,0.00,0.00,0.05,0.05
2,Pre-existing in September,With Jun–Aug WhatsApp history,5382,"4,382,663.75",814.32,710.38,30.69,"322,990.95",843.00,0.95,0.95


## 4.8 Customer-level Collections funnel

In [53]:
print("FUNNEL NESTING CHECK")
print("=" * 60)

checks = {
    "delivered_without_history": (
        historical_eligible["ever_delivered"].fillna(False)
        & ~historical_eligible["has_wa_history"].fillna(False)
    ).sum(),

    "read_without_delivery": (
        historical_eligible["ever_read"].fillna(False)
        & ~historical_eligible["ever_delivered"].fillna(False)
    ).sum(),

    "clicked_without_read": (
        historical_eligible["ever_clicked"].fillna(False)
        & ~historical_eligible["ever_read"].fillna(False)
    ).sum(),

    "replied_without_click": (
        historical_eligible["ever_replied"].fillna(False)
        & ~historical_eligible["ever_clicked"].fillna(False)
    ).sum(),

    "paid_without_reply": (
        historical_eligible["ever_paid"].fillna(False)
        & ~historical_eligible["ever_replied"].fillna(False)
    ).sum()
}

display(
    pd.Series(checks, name="customers")
    .to_frame()
)

FUNNEL NESTING CHECK


C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\709151936.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  historical_eligible["ever_delivered"].fillna(False)
C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\709151936.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  historical_eligible["ever_read"].fillna(False)
C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\709151936.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instea

,customers
delivered_without_history,0
read_without_delivery,0
clicked_without_read,461
replied_without_click,725
paid_without_reply,479


In [ ]:
- Pagamento pode ocorrer diretamente após a mensagem, sem reply;   
- reply pode ocorrer sem click;   
- click sem read pode refletir a própria definição/instrumentação desses eventos.

In [54]:
# ============================================================
# Historical WhatsApp Funnel — Eligible Population Only
# ============================================================

historical_eligible = customer.loc[
    customer["sep_entry_group"].eq("Pre-existing in September")
].copy()

n_eligible = historical_eligible["customer_id"].nunique()

funnel = pd.DataFrame({
    "stage": [
        "Historical eligible population",
        "Has Jun–Aug WA history",
        "Ever delivered",
        "Ever read",
        "Ever clicked",
        "Ever replied",
        "Ever paid within 72h"
    ],
    "customers": [
        n_eligible,
        historical_eligible["has_wa_history"].fillna(False).sum(),
        historical_eligible["ever_delivered"].fillna(False).sum(),
        historical_eligible["ever_read"].fillna(False).sum(),
        historical_eligible["ever_clicked"].fillna(False).sum(),
        historical_eligible["ever_replied"].fillna(False).sum(),
        historical_eligible["ever_paid"].fillna(False).sum()
    ]
})

# Share over historically eligible population
funnel["pct_of_eligible_population"] = (
    funnel["customers"] / n_eligible
)

# Conversion from previous funnel stage
funnel["conversion_from_previous_stage"] = (
    funnel["customers"]
    / funnel["customers"].shift(1)
)

funnel.loc[0, "conversion_from_previous_stage"] = 1.0

display(funnel)

C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\2271433099.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  historical_eligible["ever_delivered"].fillna(False).sum(),
C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\2271433099.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  historical_eligible["ever_read"].fillna(False).sum(),
C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\2271433099.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

,stage,customers,pct_of_eligible_population,conversion_from_previous_stage
0,Historical eligible population,5658,1.00,1.00
1,Has Jun–Aug WA history,5382,0.95,0.95
2,Ever delivered,5382,0.95,1.00
3,Ever read,3809,0.67,0.71
4,Ever clicked,2267,0.40,0.60
5,Ever replied,1538,0.27,0.68
6,Ever paid within 72h,762,0.13,0.50


## 4.9 Aging × exposure × historical recovery

In [56]:
# ============================================================
# Aging × Historical Recovery
# Pre-existing September population only
# ============================================================

historical_eligible = customer.loc[
    customer["sep_entry_group"].eq("Pre-existing in September")
].copy()

aging_recovery = (
    historical_eligible
    .groupby("dpd_bucket", observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        sep_outstanding_brl=("outstanding_balance_brl", "sum"),
        customers_with_wa_history=("has_wa_history", "sum"),
        historical_payers=(
            "ever_paid",
            lambda s: s.fillna(False).sum()
        ),
        historical_recovered_brl=("total_amount_paid_brl", "sum"),
        avg_sep_dpd=("days_past_due_on_2026-09-01", "mean")
    )
    .reset_index()
)

aging_recovery["customer_share"] = (
    aging_recovery["customers"]
    / aging_recovery["customers"].sum()
)

aging_recovery["sep_exposure_share"] = (
    aging_recovery["sep_outstanding_brl"]
    / aging_recovery["sep_outstanding_brl"].sum()
)

aging_recovery["historical_payer_rate"] = (
    aging_recovery["historical_payers"]
    / aging_recovery["customers_with_wa_history"].replace(0, np.nan)
)

display(aging_recovery)

C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3338272002.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


,dpd_bucket,customers,sep_outstanding_brl,customers_with_wa_history,historical_payers,historical_recovered_brl,avg_sep_dpd,customer_share,sep_exposure_share,historical_payer_rate
0,Current / 0,0,0.00,0,0,0.00,NaN,0.00,0.00,NaN
1,1–7,705,"578,210.89",450,29,"9,294.04",4.49,0.12,0.13,0.06
2,8–15,854,"702,360.67",836,91,"35,629.59",11.30,0.15,0.15,0.11
3,16–30,1432,"1,192,227.01",1429,200,"83,988.61",22.71,0.25,0.26,0.14
4,31–60,2667,"2,128,850.99",2667,442,"194,078.71",45.37,0.47,0.46,0.17
5,61–90,0,0.00,0,0,0.00,NaN,0.00,0.00,NaN
6,91+,0,0.00,0,0,0.00,NaN,0.00,0.00,NaN


## 4.10 Observed delinquency migration

In [58]:
# ============================================================
# Observed DPD Migration — Historical Eligible Population
# ============================================================

migration_population = customer.loc[
    customer["sep_entry_group"].eq("Pre-existing in September")
    & customer["has_wa_history"].fillna(False)
].copy()


migration = (
    migration_population
    .groupby(
        "dpd_migration",
        dropna=False,
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        sep_outstanding_brl=("outstanding_balance_brl", "sum"),
        historical_recovered_brl=("total_amount_paid_brl", "sum"),
        avg_initial_observed_dpd=("initial_observed_dpd", "mean"),
        avg_last_observed_dpd=("last_observed_dpd", "mean")
    )
)


migration["customer_share"] = (
    migration["customers"]
    / migration["customers"].sum()
)

migration["sep_exposure_share"] = (
    migration["sep_outstanding_brl"]
    / migration["sep_outstanding_brl"].sum()
)


display(migration)

,customers,sep_outstanding_brl,historical_recovered_brl,avg_initial_observed_dpd,avg_last_observed_dpd,customer_share,sep_exposure_share
dpd_migration,,,,,,,
Deteriorated,4966,"4,040,269.09","313,368.80",3.04,25.91,0.92,0.92
Stable,416,"342,394.66","9,622.15",3.19,3.19,0.08,0.08


- Deteriorated significa que o cliente terminou a janela observada com mais dias de atraso do que tinha no primeiro contato observado1
- Stable significa que o primeiro e o último DPD observados são iguais

## 4.11 Payment status — none / partial / full

In [61]:
# ============================================================
# Historical Payment Status — Pre-existing Population Only
# ============================================================

payment_population = customer.loc[
    customer["sep_entry_group"].eq("Pre-existing in September")
    & customer["has_wa_history"].fillna(False)
].copy()


payment_view = (
    payment_population
    .groupby(
        "payment_status",
        dropna=False,
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        historical_recovered_brl=("total_amount_paid_brl", "sum"),
        initial_observed_balance_brl=("initial_observed_balance", "sum"),
        sep_outstanding_brl=("outstanding_balance_brl", "sum")
    )
)


payment_view["customer_share"] = (
    payment_view["customers"]
    / payment_view["customers"].sum()
)


display(payment_view)

,customers,historical_recovered_brl,initial_observed_balance_brl,sep_outstanding_brl,customer_share
payment_status,,,,,
No payment,4620,0.00,"4,077,230.51","4,077,230.51",0.86
Partial payment,762,"322,990.95","628,424.17","305,433.24",0.14


## 4.12 Contact pressure

Useful for identifying possible contact fatigue / "burning the contact". Still descriptive, not causal.

In [67]:
# ============================================================
# Message Pressure Analysis
# Pre-existing Population Only
# Corrected Delivery / Failure Definition
# ============================================================


# ------------------------------------------------------------
# 1. Rebuild delivery flags at MESSAGE grain
# ------------------------------------------------------------

wa["is_delivered"] = (
    wa["delivery_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("delivered")
)

wa["is_failed"] = (
    wa["delivery_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.startswith("failed_", na=False)
)

# Failure reasons
wa["is_failed_blocked"] = (
    wa["delivery_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("failed_blocked")
)

wa["is_failed_invalid_number"] = (
    wa["delivery_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("failed_invalid_number")
)

wa["is_failed_unreachable"] = (
    wa["delivery_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("failed_unreachable")
)


# ------------------------------------------------------------
# 2. Validate raw delivery classification
# ------------------------------------------------------------

delivery_audit = pd.Series({
    "total_attempts": len(wa),
    "delivered": int(wa["is_delivered"].sum()),
    "failed": int(wa["is_failed"].sum()),
    "failed_blocked": int(wa["is_failed_blocked"].sum()),
    "failed_invalid_number": int(
        wa["is_failed_invalid_number"].sum()
    ),
    "failed_unreachable": int(
        wa["is_failed_unreachable"].sum()
    ),
    "failure_rate": wa["is_failed"].mean()
}).to_frame("value")

print("=" * 70)
print("DELIVERY STATUS AUDIT — RAW WHATSAPP HISTORY")
print("=" * 70)

display(delivery_audit)


# All attempts must be either delivered or failed
assert (
    wa["is_delivered"].sum()
    + wa["is_failed"].sum()
    == len(wa)
), "Some delivery statuses are not classified."


# Failure reasons must reconcile with total failures
assert (
    wa["is_failed_blocked"].sum()
    + wa["is_failed_invalid_number"].sum()
    + wa["is_failed_unreachable"].sum()
    == wa["is_failed"].sum()
), "Failure reasons do not reconcile with total failures."


# ------------------------------------------------------------
# 3. Reaggregate delivery metrics to CUSTOMER grain
# ------------------------------------------------------------

delivery_customer = (
    wa
    .groupby("customer_id")
    .agg(
        n_messages_delivery=("customer_id", "size"),
        n_delivered_corrected=("is_delivered", "sum"),
        n_failed_corrected=("is_failed", "sum"),
        n_failed_blocked=("is_failed_blocked", "sum"),
        n_failed_invalid_number=("is_failed_invalid_number", "sum"),
        n_failed_unreachable=("is_failed_unreachable", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Customer-level reconciliation
# ------------------------------------------------------------

assert (
    delivery_customer["n_delivered_corrected"].sum()
    + delivery_customer["n_failed_corrected"].sum()
    == delivery_customer["n_messages_delivery"].sum()
), "Customer-level delivery counts do not reconcile."


print("\nCUSTOMER-LEVEL DELIVERY RECONCILIATION")
print("-" * 70)

print(
    f"Messages    : {delivery_customer['n_messages_delivery'].sum():,}"
)

print(
    f"Delivered   : {delivery_customer['n_delivered_corrected'].sum():,}"
)

print(
    f"Failed      : {delivery_customer['n_failed_corrected'].sum():,}"
)


# ------------------------------------------------------------
# 5. Attach corrected delivery metrics to customer
# ------------------------------------------------------------
# Drop previous corrected columns if this cell is rerun

corrected_cols = [
    "n_messages_delivery",
    "n_delivered_corrected",
    "n_failed_corrected",
    "n_failed_blocked",
    "n_failed_invalid_number",
    "n_failed_unreachable"
]

customer = customer.drop(
    columns=[c for c in corrected_cols if c in customer.columns],
    errors="ignore"
)

customer = customer.merge(
    delivery_customer,
    on="customer_id",
    how="left",
    validate="1:1"
)


# Customers without historical WA records get zero counts
for col in corrected_cols:
    customer[col] = customer[col].fillna(0)


# ------------------------------------------------------------
# 6. Validate against existing n_messages
# ------------------------------------------------------------

message_count_check = (
    customer.loc[customer["has_wa_history"].fillna(False)]
    ["n_messages"]
    .eq(
        customer.loc[
            customer["has_wa_history"].fillna(False),
            "n_messages_delivery"
        ]
    )
)

print(
    "\nExisting n_messages matches reconstructed count:",
    message_count_check.all()
)


# ------------------------------------------------------------
# 7. Historical eligible population
# ------------------------------------------------------------

pressure_population = customer.loc[
    customer["sep_entry_group"].eq("Pre-existing in September")
    & customer["has_wa_history"].fillna(False)
].copy()


# ------------------------------------------------------------
# 8. Message pressure bucket
# ------------------------------------------------------------

pressure_population["message_pressure_bucket"] = pd.cut(
    pressure_population["n_messages"],
    [-0.1, 0, 1, 3, 5, 10, 20, np.inf],
    labels=[
        "0",
        "1",
        "2–3",
        "4–5",
        "6–10",
        "11–20",
        "21+"
    ]
)


# ------------------------------------------------------------
# 9. Aggregate message pressure
# ------------------------------------------------------------

pressure = (
    pressure_population
    .groupby(
        "message_pressure_bucket",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),

        sep_outstanding_brl=(
            "outstanding_balance_brl",
            "sum"
        ),

        historical_payers=(
            "ever_paid",
            lambda s: s.fillna(False).sum()
        ),

        historical_recovered_brl=(
            "total_amount_paid_brl",
            "sum"
        ),

        total_attempts=(
            "n_messages_delivery",
            "sum"
        ),

        delivered_attempts=(
            "n_delivered_corrected",
            "sum"
        ),

        failed_attempts=(
            "n_failed_corrected",
            "sum"
        ),

        failed_blocked=(
            "n_failed_blocked",
            "sum"
        ),

        failed_invalid_number=(
            "n_failed_invalid_number",
            "sum"
        ),

        failed_unreachable=(
            "n_failed_unreachable",
            "sum"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 10. Customer-level payment incidence
# ------------------------------------------------------------

pressure["payer_rate"] = (
    pressure["historical_payers"]
    / pressure["customers"].replace(0, np.nan)
)


# ------------------------------------------------------------
# 11. Attempt-level delivery metrics
# ------------------------------------------------------------

pressure["failure_rate_attempts"] = (
    pressure["failed_attempts"]
    / pressure["total_attempts"].replace(0, np.nan)
)

pressure["delivery_rate_attempts"] = (
    pressure["delivered_attempts"]
    / pressure["total_attempts"].replace(0, np.nan)
)


# ------------------------------------------------------------
# 12. Failure composition
# ------------------------------------------------------------

pressure["blocked_share_of_failures"] = (
    pressure["failed_blocked"]
    / pressure["failed_attempts"].replace(0, np.nan)
)

pressure["invalid_number_share_of_failures"] = (
    pressure["failed_invalid_number"]
    / pressure["failed_attempts"].replace(0, np.nan)
)

pressure["unreachable_share_of_failures"] = (
    pressure["failed_unreachable"]
    / pressure["failed_attempts"].replace(0, np.nan)
)


# ------------------------------------------------------------
# 13. Final reconciliation
# ------------------------------------------------------------

assert (
    pressure["total_attempts"].sum()
    ==
    pressure["delivered_attempts"].sum()
    + pressure["failed_attempts"].sum()
), "Pressure table does not reconcile attempts."


# ------------------------------------------------------------
# 14. Output
# ------------------------------------------------------------

print()
print("=" * 70)
print("MESSAGE PRESSURE — JUN TO AUG 2026")
print("=" * 70)
print(
    "Population: Pre-existing in September + Jun-Aug WA history"
)
print(
    f"Customers : "
    f"{pressure_population['customer_id'].nunique():,}"
)
print(
    f"Attempts  : "
    f"{pressure['total_attempts'].sum():,.0f}"
)
print(
    f"Delivered : "
    f"{pressure['delivered_attempts'].sum():,.0f}"
)
print(
    f"Failed    : "
    f"{pressure['failed_attempts'].sum():,.0f}"
)
print()

display(pressure)

DELIVERY STATUS AUDIT — RAW WHATSAPP HISTORY


,value
total_attempts,"75,406.00"
delivered,"63,543.00"
failed,"11,863.00"
failed_blocked,"5,051.00"
failed_invalid_number,"4,723.00"
failed_unreachable,"2,089.00"
failure_rate,0.16



CUSTOMER-LEVEL DELIVERY RECONCILIATION
----------------------------------------------------------------------
Messages    : 75,406
Delivered   : 63,543
Failed      : 11,863

Existing n_messages matches reconstructed count: True

MESSAGE PRESSURE — JUN TO AUG 2026
Population: Pre-existing in September + Jun-Aug WA history
Customers : 5,382
Attempts  : 33,347
Delivered : 27,577
Failed    : 5,770



C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\393294452.py:244: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


,message_pressure_bucket,customers,sep_outstanding_brl,historical_payers,historical_recovered_brl,total_attempts,delivered_attempts,failed_attempts,failed_blocked,failed_invalid_number,failed_unreachable,payer_rate,failure_rate_attempts,delivery_rate_attempts,blocked_share_of_failures,invalid_number_share_of_failures,unreachable_share_of_failures
0,0,0,0.00,0,0.00,0.00,0,0,0,0,0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
1,1,416,"342,394.66",28,"9,622.15",416.00,379,37,3,21,13,0.07,0.09,0.91,0.08,0.57,0.35
2,2–3,892,"731,360.38",108,"40,578.54","2,238.00",1991,247,29,152,66,0.12,0.11,0.89,0.12,0.62,0.27
3,4–5,1058,"864,559.99",169,"70,721.37","4,786.00",4134,652,123,369,160,0.16,0.14,0.86,0.19,0.57,0.25
4,6–10,2436,"1,985,264.68",360,"156,959.16","18,846.00",15643,3203,1114,1561,528,0.15,0.17,0.83,0.35,0.49,0.16
5,11–20,580,"459,084.04",97,"45,109.73","7,061.00",5430,1631,886,565,180,0.17,0.23,0.77,0.54,0.35,0.11
6,21+,0,0.00,0,0.00,0.00,0,0,0,0,0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>


## 4.13 Reusable customer-grain bivariate function

In [69]:
def collections_bivariate(df, variable):
    # Customer-grain descriptive view. Never pass raw WhatsApp messages here.
    out = df.groupby(variable, observed=False, dropna=False).agg(
        customers=("customer_id","nunique"),
        sep_outstanding_brl=("outstanding_balance_brl","sum"),
        avg_sep_balance_brl=("outstanding_balance_brl","mean"),
        customers_with_wa_history=("has_wa_history","sum"),
        historical_messages=("n_messages","sum"),
        failed_attempts=("n_failed","sum"),
        historical_recovered_brl=("total_amount_paid_brl","sum"),
        historical_payers=("ever_paid", lambda s: s.fillna(False).sum())
    ).reset_index()

    out["customer_share"] = out["customers"] / out["customers"].sum()
    out["sep_exposure_share"] = out["sep_outstanding_brl"] / out["sep_outstanding_brl"].sum()
    out["historical_payer_rate"] = (
        out["historical_payers"] /
        out["customers_with_wa_history"].replace(0,np.nan)
    )
    out["failure_rate_attempts"] = (
        out["failed_attempts"] /
        out["historical_messages"].replace(0,np.nan)
    )
    return out

display(collections_bivariate(customer, "dpd_bucket"))

C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\2258317450.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  historical_payers=("ever_paid", lambda s: s.fillna(False).sum())


,dpd_bucket,customers,sep_outstanding_brl,avg_sep_balance_brl,customers_with_wa_history,historical_messages,failed_attempts,historical_recovered_brl,historical_payers,customer_share,sep_exposure_share,historical_payer_rate,failure_rate_attempts
0,Current / 0,5000,"4,284,358.42",856.87,0,0.00,0.00,0.00,0,0.47,0.48,NaN,NaN
1,1–7,705,"578,210.89",820.16,450,672.00,0.00,"9,294.04",29,0.07,0.07,0.06,0.00
2,8–15,854,"702,360.67",822.44,836,"2,576.00",0.00,"35,629.59",91,0.08,0.08,0.11,0.00
3,16–30,1432,"1,192,227.01",832.56,1429,"7,763.00",0.00,"83,988.61",200,0.13,0.13,0.14,0.00
4,31–60,2667,"2,128,850.99",798.22,2667,"22,336.00",0.00,"194,078.71",442,0.25,0.24,0.17,0.00
5,61–90,0,0.00,NaN,0,0.00,0.00,0.00,0,0.00,0.00,NaN,NaN
6,91+,0,0.00,NaN,0,0.00,0.00,0.00,0,0.00,0.00,NaN,NaN


## 4.14 Recommended first bivariate cuts

In [71]:
def collections_bivariate(df, variable):
    """
    Customer-grain bivariate Collections analysis.

    Always splits the September portfolio into:
    - Pre-existing in September
    - Entered in September

    Historical Jun-Aug metrics are only valid for customers
    who were already in Collections before September.

    Never pass raw WhatsApp message-level data here.
    """

    # --------------------------------------------------------
    # 1. Keep the two analytical September populations
    # --------------------------------------------------------

    analysis_df = df.loc[
        df["sep_entry_group"].isin([
            "Pre-existing in September",
            "Entered in September"
        ])
    ].copy()


    # --------------------------------------------------------
    # 2. Aggregate
    # --------------------------------------------------------

    out = (
        analysis_df
        .groupby(
            ["sep_entry_group", variable],
            observed=False,
            dropna=False
        )
        .agg(
            customers=(
                "customer_id",
                "nunique"
            ),

            sep_outstanding_brl=(
                "outstanding_balance_brl",
                "sum"
            ),

            avg_sep_balance_brl=(
                "outstanding_balance_brl",
                "mean"
            ),

            customers_with_wa_history=(
                "has_wa_history",
                lambda s: s.fillna(False).sum()
            ),

            historical_messages=(
                "n_messages",
                "sum"
            ),

            failed_attempts=(
                "n_failed_corrected",
                "sum"
            ),

            historical_recovered_brl=(
                "total_amount_paid_brl",
                "sum"
            ),

            historical_payers=(
                "ever_paid",
                lambda s: s.fillna(False).sum()
            )
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # 3. Shares WITHIN each September population
    # --------------------------------------------------------

    out["customer_share"] = (
        out["customers"]
        / out.groupby("sep_entry_group")["customers"].transform("sum")
    )

    out["sep_exposure_share"] = (
        out["sep_outstanding_brl"]
        / out.groupby("sep_entry_group")["sep_outstanding_brl"].transform("sum")
    )


    # --------------------------------------------------------
    # 4. Historical performance
    #    Only meaningful for Pre-existing population
    # --------------------------------------------------------

    out["historical_payer_rate"] = (
        out["historical_payers"]
        / out["customers_with_wa_history"].replace(0, np.nan)
    )

    out["failure_rate_attempts"] = (
        out["failed_attempts"]
        / out["historical_messages"].replace(0, np.nan)
    )


    # --------------------------------------------------------
    # 5. Structural NA for September entrants
    # --------------------------------------------------------
    # These customers were not historically eligible in Jun-Aug.
    # Zero would incorrectly imply observed absence of activity.

    historical_cols = [
        "customers_with_wa_history",
        "historical_messages",
        "failed_attempts",
        "historical_recovered_brl",
        "historical_payers",
        "historical_payer_rate",
        "failure_rate_attempts"
    ]

    entered_sep = (
        out["sep_entry_group"]
        .eq("Entered in September")
    )

    out.loc[
        entered_sep,
        historical_cols
    ] = np.nan


    return out

In [72]:
bivariate_variables = [
    "dpd_bucket",
    "balance_bucket",
    "salary_bucket",
    "state_uf",
    "contact_history_group",
    "message_pressure_bucket",
    "dpd_migration",
    "payment_status"
]


for variable in bivariate_variables:

    print("\n" + "=" * 100)
    print(variable.upper())
    print("=" * 100)

    display(
        collections_bivariate(
            customer,
            variable
        )
    )


DPD_BUCKET


C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3477774492.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


,sep_entry_group,dpd_bucket,customers,sep_outstanding_brl,avg_sep_balance_brl,customers_with_wa_history,historical_messages,failed_attempts,historical_recovered_brl,historical_payers,customer_share,sep_exposure_share,historical_payer_rate,failure_rate_attempts
0,Entered in September,Current / 0,5000,"4,284,358.42",856.87,NaN,NaN,<NA>,NaN,NaN,1.00,1.00,NaN,<NA>
1,Entered in September,1–7,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
2,Entered in September,8–15,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
3,Entered in September,16–30,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
4,Entered in September,31–60,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
5,Entered in September,61–90,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
6,Entered in September,91+,0,0.00,NaN,NaN,NaN,<NA>,NaN,NaN,0.00,0.00,NaN,<NA>
7,Pre-existing in September,Current / 0,0,0.00,NaN,NaN,0.00,0,0.00,NaN,0.00,0.00,NaN,<NA>
8,Pre-existing in September,1–7,705,"578,210.89",820.16,450.00,672.00,68,"9,294.04",29.00,0.12,0.13,0.06,0.10
9,Pre-existing in September,8–15,854,"702,360.67",822.44,836.00,"2,576.00",258,"35,629.59",91.00,0.15,0.15,0.11,0.10



BALANCE_BUCKET


C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3477774492.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


...

C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3477774492.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


,sep_entry_group,dpd_migration,customers,sep_outstanding_brl,avg_sep_balance_brl,customers_with_wa_history,historical_messages,failed_attempts,historical_recovered_brl,historical_payers,customer_share,sep_exposure_share,historical_payer_rate,failure_rate_attempts
0,Entered in September,NaN,5000,"4,284,358.42",856.87,NaN,NaN,<NA>,NaN,NaN,1.00,1.00,NaN,<NA>
1,Pre-existing in September,Deteriorated,4966,"4,040,269.09",813.59,"4,966.00","32,931.00",5733,"313,368.80",734.00,0.88,0.88,0.15,0.17
2,Pre-existing in September,Stable,416,"342,394.66",823.06,416.00,416.00,37,"9,622.15",28.00,0.07,0.07,0.07,0.09
3,Pre-existing in September,NaN,276,"218,985.81",793.43,0.00,0.00,0,0.00,0.00,0.05,0.05,NaN,<NA>



PAYMENT_STATUS


C:\Users\beelt\AppData\Local\Temp\ipykernel_20596\3477774492.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda s: s.fillna(False).sum()


,sep_entry_group,payment_status,customers,sep_outstanding_brl,avg_sep_balance_brl,customers_with_wa_history,historical_messages,failed_attempts,historical_recovered_brl,historical_payers,customer_share,sep_exposure_share,historical_payer_rate,failure_rate_attempts
0,Entered in September,NaN,5000,"4,284,358.42",856.87,NaN,NaN,<NA>,NaN,NaN,1.00,1.00,NaN,<NA>
1,Pre-existing in September,No payment,4620,"4,077,230.51",882.52,"4,620.00","28,329.00",5467,0.00,0.00,0.82,0.89,0.00,0.19
2,Pre-existing in September,Partial payment,762,"305,433.24",400.83,762.00,"5,018.00",303,"322,990.95",762.00,0.13,0.07,1.00,0.06
3,Pre-existing in September,NaN,276,"218,985.81",793.43,0.00,0.00,0,0.00,0.00,0.05,0.05,NaN,<NA>


## 4.15 Executive integrity checks

In [73]:
checks = pd.Series({
    "queue_unique_customer_grain": queue["customer_id"].is_unique,
    "wa_customer_unique_grain": wa_customer["customer_id"].is_unique,
    "final_customer_unique_grain": customer["customer_id"].is_unique,
    "final_rows_equal_queue_rows": len(customer) == len(queue),
    "sep_exposure_preserved_after_merge": np.isclose(
        customer["outstanding_balance_brl"].sum(),
        queue["outstanding_balance_brl"].sum()
    ),
    "no_negative_message_counts": (customer["n_messages"] >= 0).all()
}).to_frame("passed")

display(checks)
assert checks["passed"].all()
print("ALL COLLECTIONS GRAIN / EXPOSURE CHECKS PASSED")

,passed
queue_unique_customer_grain,True
wa_customer_unique_grain,True
final_customer_unique_grain,True
final_rows_equal_queue_rows,True
sep_exposure_preserved_after_merge,True
no_negative_message_counts,True


ALL COLLECTIONS GRAIN / EXPOSURE CHECKS PASSED
